In [0]:
from pyspark.sql.functions import col

df_bronze = spark.table("assesment.bronze.ons_very_old_population_raw")

# Get the header row (the row where _c2 says "90 & over")
header_row = df_bronze.filter(col("_c2") == "90 & over").first()

# Rename _c2 to _c20 one by one, using the labels from the header row
df_step1 = df_bronze
for old_name in df_bronze.columns[2:21]:
    label = str(header_row[old_name])                         # e.g. "90 & over"
    new_name = label.replace("&", "and").replace(" ", "_")    # e.g. "90_and_over"
    df_step1 = df_step1.withColumnRenamed(old_name, new_name)

display(df_step1)

In [0]:
# Step 2: rename the first two columns
df_step2 = (
    df_step1
    .withColumnRenamed("_c0", "population")
    .withColumnRenamed("_c1", "year")
)

display(df_step2)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

# Step 3: remove the bottom 7 rows (notes)
total_rows = df_step2.count()

df_step3 = (
    df_step2
    .withColumn("row_num", row_number().over(Window.orderBy("_source_row_id")))  # number rows 1..78 in file order
    .filter(col("row_num") <= total_rows - 7)                                    # keep rows 1..71
    .drop("row_num")
)

print(f"Rows before: {total_rows}, after: {df_step3.count()}")   # 78 -> 71
display(df_step3.orderBy("_source_row_id"))

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, last, row_number

# Read from bronze
df_bronze = spark.table("assesment.bronze.ons_very_old_population_raw")

# Step 1: rename _c2.._c20 using the header row (the row where _c2 says "90 & over")
header_row = df_bronze.filter(col("_c2") == "90 & over").first()
df_step1 = df_bronze
for old_name in df_bronze.columns[2:21]:
    new_name = str(header_row[old_name]).replace("&", "and").replace(" ", "_")
    df_step1 = df_step1.withColumnRenamed(old_name, new_name)

# Step 2: rename _c0 and _c1
df_step2 = df_step1.withColumnRenamed("_c0", "population").withColumnRenamed("_c1", "year")

# Step 3: remove the bottom 7 rows (notes)
total_rows = df_step2.count()
df_step3 = (
    df_step2
    .withColumn("row_num", row_number().over(Window.orderBy("_source_row_id")))
    .filter(col("row_num") <= total_rows - 7)
    .drop("row_num")
)

# Step 4: fill population down, keep only data rows, drop audit columns
fill_down = Window.orderBy("_source_row_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_step4 = (
    df_step3
    .withColumn("population", last("population", ignorenulls=True).over(fill_down))
    .filter(col("year").isNotNull())
    .drop("_source_row_id", "_source_file", "_ingested_at")
)

# Split into 3 DataFrames
df_persons = df_step4.filter(col("population") == "Persons").orderBy("year")
df_males = df_step4.filter(col("population") == "Males").orderBy("year")
df_females = df_step4.filter(col("population") == "Females").orderBy("year")

# Save to silver
spark.sql("CREATE SCHEMA IF NOT EXISTS assesment.silver")
df_persons.write.format("delta").mode("overwrite").saveAsTable("assesment.silver.df_persons")
df_males.write.format("delta").mode("overwrite").saveAsTable("assesment.silver.df_males")
df_females.write.format("delta").mode("overwrite").saveAsTable("assesment.silver.df_females")

print(f"Persons: {df_persons.count()}, Males: {df_males.count()}, Females: {df_females.count()}")   # 21 each

# Step 5: display Persons
display(df_persons)

In [0]:
display(df_persons)
